## 데이터 전처리 파이프라인 주요 수정 및 고도화 내역
### 텍스트 정제 및 청킹 프로세스 통합함
- 기존에는 특수문자 등을 제거하는 정제 단계(Preprocessing_Cleaning.ipynb)와 텍스트를 분할하는 단계(text_chunking.ipynb)가 분리되어 있었음.

- clean_and_chunk_with_meta 함수를 생성하여, 텍스트 정규표현식 정제 후 곧바로 길이 기반 청킹이 이루어지도록 단일 파이프라인으로 통합함.

### 청크(Chunk) 단위 메타데이터 주입 기능 추가함
- 쪼개진 텍스트 조각만으로는 AI가 맥락을 파악하기 어렵다는 점을 보완함.

- 각 조각 최상단에 [발주기관 | 사업명 | 금액] 형태의 메타데이터(이름표)를 강제 주입하는 로직을 추가함.

### 메타데이터 세부 정보 및 안정성 고도화함
- 결측치(NaN) 방어 로직 적용: 데이터가 비어있을 경우 'nan' 텍스트나 에러가 발생하는 것을 막기 위해, 빈 값을 '미상'으로 안전하게 치환하는 safe_str 헬퍼 함수를 적용함.

- 핵심 검색 단서 추가: 복구했던 데이터인 '공고 번호'와 '입찰 참여 시작일'을 메타데이터 항목에 추가하여 검색 정밀도를 높임.

- 조각 순서(Index) 표기: 분할된 문서의 전후 맥락 유지를 위해 조각순서: 1/5 형태로 현재 청크의 위치 정보를 메타데이터에 포함함.

### 구조 기반 청킹(심화) 적용 보류함
- 공통 섹션 구조(목차 패턴 등)를 정규표현식으로 인식하여 자르는 '구조 기반 청킹'은 현 단계의 오버엔지니어링으로 판단하여 제외함.

- 기존에 작성한 글자 수 기반의 RecursiveCharacterTextSplitter 방식을 유지하여 빠르고 안정적인 분할을 우선시함.

In [1]:
import re
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("🚀 [Step 2] 정제 ➡️ 청킹 ➡️ 메타데이터 주입 통합 파이프라인 시작...\n")

# 1. 원본 데이터 불러오기
file_path = '/home/bidcoin/bid_master_cleaned.csv'  # 나중에 복구된 데이터 파일 사용
df = pd.read_csv(file_path)
print(f"✅ 원본 데이터: 총 {len(df)}건\n")

# 통합된 클리닝 & 청킹 & 메타데이터 주입 함수
def clean_and_chunk_with_meta(row):
    raw_text = row['텍스트']
    
    if pd.isna(raw_text) or len(str(raw_text)) == 0:
        return []
    
    # ------------------------------------------------------------
    # 🧹 [Step 1] 텍스트 정제
    # ------------------------------------------------------------
    cleaned_text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@]', ' ', str(raw_text))
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    cleaned_len = len(cleaned_text)
    
    # ------------------------------------------------------------
    # ✂️ [Step 2] 맞춤형 청킹
    # ------------------------------------------------------------
    if cleaned_len < 500:
        chunk_size = 500
        chunk_overlap = 0
    elif cleaned_len < 3000:
        chunk_size = 512
        chunk_overlap = 50
    elif cleaned_len < 10000:
        chunk_size = 512
        chunk_overlap = 100
    else:
        chunk_size = 1024
        chunk_overlap = 200

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(cleaned_text)
    
    # ------------------------------------------------------------
    # 🏷️ [Step 3] 메타데이터 주입 (디테일 업그레이드!)
    # ------------------------------------------------------------
    chunks_with_meta = []
    
    # 결측치(NaN)를 '미상' 또는 빈 문자열로 안전하게 변환하는 헬퍼 함수
    def safe_str(val):
        return "미상" if pd.isna(val) else str(val)
    
    # 금액 포맷팅 안전장치
    try:
        amt_str = f"{float(row['사업 금액']):,.0f}원" if not pd.isna(row['사업 금액']) else "미상"
    except:
        amt_str = safe_str(row['사업 금액'])
        
    # 날짜와 공고번호 가져오기 (컬럼이 없을 경우를 대비해 get 메서드 사용)
    notice_id = safe_str(row.get('공고 번호', '미상'))
    start_date = safe_str(row.get('입찰 참여 시작일', '미상'))
    
    total_chunks = len(chunks)
    
    for i, chunk_text in enumerate(chunks):
        # 조각 순서(i+1/total)와 날짜/공고번호 추가
        meta_header = (
            f"[공고번호: {notice_id} | 발주기관: {safe_str(row['발주 기관'])} | "
            f"사업명: {safe_str(row['사업명'])} | 금액: {amt_str} | "
            f"시작일: {start_date} | 조각순서: {i+1}/{total_chunks}]"
        )
        chunk_with_meta = f"{meta_header}\n\n{chunk_text}"
        chunks_with_meta.append(chunk_with_meta)
        
    return chunks_with_meta

# 3. 데이터프레임에 통합 함수 적용
print("✂️ 텍스트 정제, 청킹, 이름표 부착을 동시에 진행합니다. 잠시만 기다려주세요...")
df['청크_리스트'] = df.apply(clean_and_chunk_with_meta, axis=1)

# 4. 쪼개진 리스트를 각각의 행(Row)으로 분리 (데이터 폭발! 💥)
chunked_df = df.explode('청크_리스트').reset_index(drop=True)

# 5. 컬럼 정리 및 새로운 텍스트 길이 계산
chunked_df = chunked_df.rename(columns={'청크_리스트': '청크_텍스트'})

# 원본 '텍스트'와 '텍스트길이' 컬럼은 삭제하여 메모리 확보 (에러 방지용 errors='ignore' 추가)
chunked_df = chunked_df.drop(columns=['텍스트', '텍스트길이'], errors='ignore')

# 최종 청크_텍스트의 글자 수 다시 세기
chunked_df['청크_길이'] = chunked_df['청크_텍스트'].apply(lambda x: len(x) if isinstance(x, str) else 0)

# 6. 최종 완성 데이터 저장
# 본인 경로로 수정하기
save_path = '/home/bidcoin/bid_master_chunked.csv'
chunked_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print("-" * 60)
print(f"✨ 작업 완료! 원본 {len(df)}건의 문서가 총 {len(chunked_df)}개의 완벽한 조각으로 나누어졌습니다.")
print(f"💾 결과물이 '{save_path}'에 안전하게 저장되었습니다!")

# 결과 샘플 확인
display(chunked_df[['파일명', '청크_텍스트', '청크_길이']].head())

🚀 [Step 2] 정제 ➡️ 청킹 ➡️ 메타데이터 주입 통합 파이프라인 시작...

✅ 원본 데이터: 총 100건

✂️ 텍스트 정제, 청킹, 이름표 부착을 동시에 진행합니다. 잠시만 기다려주세요...
------------------------------------------------------------
✨ 작업 완료! 원본 100건의 문서가 총 960개의 완벽한 조각으로 나누어졌습니다.
💾 결과물이 '/home/bidcoin/bid_master_chunked.csv'에 안전하게 저장되었습니다!


,파일명,청크_텍스트,청크_길이
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,471
1,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,634
2,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,341
3,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,[공고번호: 20241001798 | 발주기관: 한영대학 | 사업명: 한영대학교 특...,300
4,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,[공고번호: 20241002912 | 발주기관: 한국연구재단 | 사업명: 2024년...,544


In [ ]:
df = pd.read_csv(file_path)

raw_text = row['텍스트']
 
if pd.isna(raw_text) or len(str(raw_text)) == 0:
    return []
 
# ------------------------------------------------------------
# 🧹 [Step 1] 텍스트 정제
# ------------------------------------------------------------
cleaned_text = re.sub(r'[^가-힣a-zA-Z0-9\s\.\(\)\[\]\/\,\%\:\-\·\?\!\@]', ' ', str(raw_text))
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
cleaned_len = len(cleaned_text)